In [26]:
# Prompt Templates
# They help to turn raw user information into a format that the llm can work with.
# In this case the raw user input is just a message, which we are passing to the llm. Let's now make that a bit more complicated. First let's add in a system message with some custom instructions (but still taking messages as input). Next we'll add in more input besides just the messages.

In [27]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [28]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage

In [29]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="groq/compound",groq_api_key=groq_api_key)
llm

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x00000289E93C0B90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000289E93C21E0>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [30]:
llm.invoke([HumanMessage(content="Hello, how are you?")])

AIMessage(content="Hello! I'm doing great, thank you. How can I help you today?", additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 242, 'total_tokens': 299, 'completion_time': 0.119789, 'completion_tokens_details': None, 'prompt_time': 0.007527, 'prompt_tokens_details': None, 'queue_time': 0.109074, 'total_time': 0.127316}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--60bed846-d25e-4dc4-9725-81336bd7ce15-0', usage_metadata={'input_tokens': 242, 'output_tokens': 57, 'total_tokens': 299})

In [39]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages([
("system",
        "You are a helpful assistant. Answer all the questions to the best of your ability. "
        "Always respond in this language: {language}."),
MessagesPlaceholder(variable_name="messages")
])
chain = prompt | llm

In [32]:
chain.invoke({
    "messages":[HumanMessage(content="Hi my name is John. ")]
} )

AIMessage(content='Hello John! Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 311, 'total_tokens': 369, 'completion_time': 0.138329, 'completion_tokens_details': None, 'prompt_time': 0.010421, 'prompt_tokens_details': None, 'queue_time': 0.097132, 'total_time': 0.14875}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--8eca7525-5a8d-4e34-839d-4b48666d526b-0', usage_metadata={'input_tokens': 311, 'output_tokens': 58, 'total_tokens': 369})

In [33]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


In [34]:
store = {}

In [35]:
def get_session_history(session_id: str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]= ChatMessageHistory()
    return store[session_id]

In [36]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

In [37]:
config = {"configurable":{"session_id":"chat3"}}
response = with_message_history.invoke(HumanMessage(content="Hi my name is RIa"),config=config)
response.content

'Hello\u202fRIa! 👋  \n\nYour first message was simply a friendly introduction (“Hi my name is\u202fRIa”), which isn’t a question that requires a factual answer. Recognizing that, I responded with a warm greeting and let you know I’m here to help.  \n\nIf you have any specific question, topic you’d like to discuss, or anything you need assistance with, just let me know—I’m ready to dive in!'

In [41]:
response = chain.invoke({"messages":[HumanMessage(content="How are you")],"language":"Hindi"})
response.content

'मैं ठीक हूँ, धन्यवाद! आप कैसे हैं?'

In [42]:
# let us now wrap this more complicated chain in a msg history class.Because there are multiple keys in the input, we need to specify the correct key to use to save the chat history

In [43]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="messages")

In [44]:
config = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {"messages":[HumanMessage(content="Hello, my name is Ria")],"language":"Spanish"},
    config=config
)
response.content

'¡Hola, Ria! Encantado de conocerte. ¿En qué puedo ayudarte hoy?'